# 🚀 Adaptive AI Project Manager — GRPO Training

**OpenEnv Hackathon India 2026 | Round 2**

This notebook trains an LLM using GRPO on the Agile sprint simulation environment.

**Plots produced:**
- Training loss + reward curves
- Before vs After baseline comparison
- Per-rubric reward breakdown
- Episode reward improvement over time

> ⚠️ Set Runtime → T4 GPU before running

In [1]:
# Common dependency setup for both Colab and Kaggle
# NOTE: Run this cell first and check output for any red errors.

import subprocess
import sys
import os

IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
IS_COLAB = "COLAB_RELEASE_TAG" in os.environ or "google.colab" in sys.modules
runtime = "Kaggle" if IS_KAGGLE else "Colab" if IS_COLAB else "Other"
print(f"Runtime detected: {runtime}")


def run_pip(pkg):
    print(f"Installing {pkg} ...")
    r = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", pkg],
        capture_output=True,
        text=True,
    )
    if r.returncode != 0:
        print(f"ERROR installing {pkg}")
        if r.stderr:
            print(r.stderr[-2500:])
        if r.stdout:
            print(r.stdout[-1000:])
    else:
        print(f"  OK: {pkg}")
    return r.returncode == 0

# Same dependency intent for both runtimes; only unsloth install order differs.
if IS_KAGGLE:
    unsloth_candidates = [
        "unsloth",
        "unsloth @ git+https://github.com/unslothai/unsloth.git",
    ]
else:
    unsloth_candidates = [
        "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git",
        "unsloth",
    ]

common_packages = [
    "trl>=0.15.0",
    "openenv-core>=0.2.0",
    "requests>=2.31.0",
    "datasets>=2.18.0",
    "transformers>=4.40",
    "accelerate>=0.30.0",
    "peft>=0.11.0",
    "bitsandbytes>=0.43.0",
    "matplotlib>=3.7.0",
    "pandas>=2.0.0",
    "numpy>=1.24.0",
]

all_ok = True

# Install unsloth with fallback.
unsloth_ok = False
for pkg in unsloth_candidates:
    if run_pip(pkg):
        unsloth_ok = True
        break
if not unsloth_ok:
    all_ok = False

# Install common packages.
for pkg in common_packages:
    if not run_pip(pkg):
        all_ok = False

if not all_ok:
    print("\nSome packages failed to install. See errors above.")
    print("If on Kaggle/Colab, ensure GPU is enabled and retry.")
else:
    print("\nAll pip installations completed.")

# Verify imports.
print("\n--- Verifying imports ---")
imports = [
    ("torch", "PyTorch"),
    ("transformers", "Transformers"),
    ("datasets", "Datasets"),
    ("trl", "TRL"),
    ("peft", "PEFT"),
    ("unsloth", "Unsloth"),
    ("matplotlib", "Matplotlib"),
    ("pandas", "Pandas"),
    ("numpy", "NumPy"),
    ("requests", "Requests"),
]

missing = []
for mod, name in imports:
    try:
        __import__(mod)
        print(f"  OK: {name}")
    except ImportError as e:
        print(f"  MISSING: {name}: {e}")
        missing.append(name)

if missing:
    raise ImportError(f"Missing required packages: {missing}")

print("\nAll dependencies installed and verified.")
print("If first-time install, restart runtime/session once before training.")


Runtime detected: Kaggle
Installing unsloth ...
  OK: unsloth
Installing trl>=0.15.0 ...
  OK: trl>=0.15.0
Installing openenv-core>=0.2.0 ...
  OK: openenv-core>=0.2.0
Installing requests>=2.31.0 ...
  OK: requests>=2.31.0
Installing datasets>=2.18.0 ...
  OK: datasets>=2.18.0
Installing transformers>=4.40 ...
  OK: transformers>=4.40
Installing accelerate>=0.30.0 ...
  OK: accelerate>=0.30.0
Installing peft>=0.11.0 ...
  OK: peft>=0.11.0
Installing bitsandbytes>=0.43.0 ...
  OK: bitsandbytes>=0.43.0
Installing matplotlib>=3.7.0 ...
  OK: matplotlib>=3.7.0
Installing pandas>=2.0.0 ...
  OK: pandas>=2.0.0
Installing numpy>=1.24.0 ...
  OK: numpy>=1.24.0

All pip installations completed.

--- Verifying imports ---
  OK: PyTorch
  OK: Transformers
  OK: Datasets
  OK: TRL


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


  OK: PEFT
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/tmp/ipykernel_55/2068260923.py:97: UserWarning: WARNING: Unsloth should be imported before [trl, transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  __import__(mod)


🦥 Unsloth Zoo will now patch everything to make training faster!
  OK: Unsloth
  OK: Matplotlib
  OK: Pandas
  OK: NumPy
  OK: Requests

All dependencies installed and verified.
If first-time install, restart runtime/session once before training.


## Cell 2 — Configuration
> 🔧 Set your HF Space URL below

In [2]:
import sys
import json, os, random, re, requests
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

# ─────────────────────────────────────────────
# 🔧 SET YOUR HF SPACE URL HERE
OPENENV_BASE_URL = "https://dharshansriram-openenv-pr-ai.hf.space"  # ← replace
HF_REPO_ID       = "dharshansriram/openenv-pr-ai"
# ─────────────────────────────────────────────

BASE_MODEL   = "unsloth/Qwen2.5-1.5B-Instruct"
OUTPUT_DIR   = "/kaggle/working/pm-grpo-checkpoints" if IS_KAGGLE else "/content/pm-grpo-checkpoints"
MAX_EPISODES = 50      # increase to 200 for full training
EVAL_SEEDS   = 5
SCENARIOS    = ["easy", "medium", "hard"]
MAX_STEPS_EP = 40

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"✅ Config ready")
print(f"   Runtime     : {'Kaggle' if IS_KAGGLE else 'Colab' if IS_COLAB else 'Other'}")
print(f"   Environment : {OPENENV_BASE_URL}")
print(f"   Base model  : {BASE_MODEL}")
print(f"   Output dir  : {OUTPUT_DIR}")
print(f"   Episodes    : {MAX_EPISODES}")

✅ Config ready
   Runtime     : Kaggle
   Environment : https://dharshansriram-openenv-pr-ai.hf.space
   Base model  : unsloth/Qwen2.5-1.5B-Instruct
   Output dir  : /kaggle/working/pm-grpo-checkpoints
   Episodes    : 50


## Cell 3 — Connect to Environment

In [3]:
class PMEnvClient:
    def __init__(self, base_url=OPENENV_BASE_URL):
        self.base = base_url.rstrip("/")

    def reset(self, scenario="medium", seed=42):
        r = requests.post(f"{self.base}/reset",
                          json={"scenario": scenario, "seed": seed}, timeout=30)
        r.raise_for_status()
        return r.json()

    def step(self, session_id, action):
        r = requests.post(f"{self.base}/step",
                          json={"session_id": session_id, **action}, timeout=30)
        r.raise_for_status()
        return r.json()

    def grade(self, session_id):
        r = requests.get(f"{self.base}/grader",
                         params={"session_id": session_id}, timeout=30)
        r.raise_for_status()
        return r.json()

    def demo(self, scenario="medium", seed=42):
        r = requests.get(f"{self.base}/demo",
                         params={"scenario": scenario, "seed": seed}, timeout=60)
        r.raise_for_status()
        return r.json()

env_client = PMEnvClient()

try:
    h = requests.get(f"{OPENENV_BASE_URL}/health", timeout=10)
    h.raise_for_status()
    print(f"✅ Environment is UP at {OPENENV_BASE_URL}")
except Exception as e:
    print(f"❌ Cannot reach environment: {e}")
    print("   → Deploy your HF Space first, then update OPENENV_BASE_URL above")

✅ Environment is UP at https://dharshansriram-openenv-pr-ai.hf.space


## Cell 4 — Baseline Scores (BEFORE Training)

In [4]:
def get_baseline_scores(n_seeds=EVAL_SEEDS):
    baseline = {}
    for scenario in SCENARIOS:
        scores = []
        for seed in range(n_seeds):
            try:
                data = env_client.demo(scenario=scenario, seed=seed)["data"]
                scores.append(data.get("weighted_total", 0.0))
            except Exception as e:
                print(f"  baseline {scenario} seed={seed}: {e}")
                scores.append(0.0)
        baseline[scenario] = scores
        mean = sum(scores) / len(scores)
        print(f"  {scenario:<8} | mean={mean:.3f} | scores={[round(s,3) for s in scores]}")
    return baseline

print("Collecting BASELINE scores (heuristic agent)...")
print("-" * 55)
baseline_scores = get_baseline_scores()
print("\n✅ Baseline captured — we will beat these after training")

-------------------------------------------------------
  easy     | mean=0.769 | scores=[0.64, 0.877, 0.815, 0.789, 0.723]
  medium   | mean=0.668 | scores=[0.532, 0.666, 0.687, 0.791, 0.664]
  hard     | mean=0.741 | scores=[0.739, 0.704, 0.735, 0.731, 0.798]

✅ Baseline captured — we will beat these after training


## Cell 5 — Load Model with Unsloth (4-bit QLoRA)

In [5]:
import torch
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = BASE_MODEL,
    max_seq_length = 1024,
    load_in_4bit   = True,
    dtype          = None,
)

model = FastLanguageModel.get_peft_model(
    model,
    r                          = 16,
    target_modules             = ["q_proj", "v_proj", "k_proj", "o_proj",
                                   "gate_proj", "up_proj", "down_proj"],
    lora_alpha                 = 32,
    lora_dropout               = 0.0,
    bias                       = "none",
    use_gradient_checkpointing = "unsloth",
    random_state               = 42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"✅ Model loaded with Unsloth 4-bit QLoRA")
print(f"   Trainable: {trainable:,} ({100*trainable/total:.2f}%)")
print(f"   Total    : {total:,}")

==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.53G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/qwen2.5-1.5b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.4.8 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


✅ Model loaded with Unsloth 4-bit QLoRA
   Trainable: 18,464,768 (1.78%)
   Total    : 1,036,449,280


## Cell 6 — Prompt Builder + Reward Function

**Important:** This cell is markdown only (no code). The logic lives in the **next cell** (Python): `obs_to_prompt`, `parse_action`, and `grpo_reward_fn`. After any edit, **run that Python cell** before dataset build, training, or eval.

In [6]:
def obs_to_prompt(obs, scenario):
    step      = obs.get("step", 0)
    max_steps = obs.get("max_steps", 30)
    metrics   = obs.get("metrics", {})
    tasks     = obs.get("tasks", [])
    devs      = obs.get("developers", [])
    events    = obs.get("recent_events", [])

    task_lines = []
    for t in sorted(tasks, key=lambda x: (
        0 if x["status"] in ("ready", "in_progress") else 1,
        -x.get("business_value", 0),
    ))[:6]:
        task_lines.append(
            f"  {t['id']} | {t['name'][:18]:<18} | {t['status']:<11} | "
            f"pri={t['priority']:<8} | {t.get('remaining_points',0):.1f}sp | "
            f"dl={t['deadline_step']} | skills={t.get('required_skills',[])}")

    dev_lines = []
    for d in devs:
        top = ", ".join(
            f"{k}={v:.2f}" for k, v in
            sorted((d.get("skills") or {}).items(), key=lambda x: -x[1])[:2])
        dev_lines.append(
            f"  {d['id']} | {d['name']:<12} | fatigue={d['fatigue']:.2f} | "
            f"avail={d['available']} | busy={d.get('current_tasks',[])} | {top}")

    ev = "\n".join(f"  ⚡ {e['description']}" for e in events[-2:]) or "  None"

    lines = [
        "You are an AI Agile project manager. Maximise sprint delivery.",
        "",
        f"SPRINT: step={step}/{max_steps} scenario={scenario}",
        f"Progress: {metrics.get('delivered_story_points',0):.1f}/{metrics.get('total_story_points',0):.1f} SP",
        "",
        "TASKS:",
        *task_lines,
        "",
        "DEVELOPERS:",
        *dev_lines,
        "",
        f"EVENTS: {ev}",
        "",
        "Respond with ONE JSON object only (no markdown, no prose). Valid shapes:",
        '{"action_type":"assign_task","task_id":"<id>","developer_id":"<id>"}',
        '{"action_type":"unassign_task","task_id":"<id>","developer_id":"<id>"}',
        '{"action_type":"reprioritize","task_id":"<id>","new_priority":"CRITICAL"}',
        '  (new_priority must be LOW, MEDIUM, HIGH, or CRITICAL — uppercase)',
        '{"action_type":"rest_developer","developer_id":"<id>"}',
        '{"action_type":"split_task","task_id":"<id>","split_ratio":0.5}',
        '{"action_type":"pair_program","task_id":"<id>","primary_developer_id":"<id>","secondary_developer_id":"<id>"}',
        '{"action_type":"noop"}',
        "",
        "JSON:",
    ]
    return "\n".join(lines)


def _strip_json_fences(text):
    text = re.sub(r"```(?:json)?\s*", "", text, flags=re.IGNORECASE)
    return text.replace("```", "").strip()


def _smart_quotes_to_ascii(text):
    return (
        text.replace("\u201c", '"')
        .replace("\u201d", '"')
        .replace("\u2018", "'")
        .replace("\u2019", "'")
    )


def _balanced_json_slices(text):
    """Yield every top-level {...} substring (handles nested braces)."""
    n = len(text)
    for i, ch in enumerate(text):
        if ch != "{":
            continue
        depth = 0
        in_str = False
        esc = False
        quote = None
        for j in range(i, n):
            c = text[j]
            if in_str:
                if esc:
                    esc = False
                elif c == "\\":
                    esc = True
                elif c == quote:
                    in_str = False
                continue
            if c in ('"', "'"):
                in_str = True
                quote = c
                continue
            if c == "{":
                depth += 1
            elif c == "}":
                depth -= 1
                if depth == 0:
                    yield text[i : j + 1]
                    break


def _unwrap_nested_action(d):
    if not isinstance(d, dict):
        return d
    for k in ("action", "decision", "tool_call", "output", "response"):
        inner = d.get(k)
        if isinstance(inner, dict) and "action_type" in inner:
            return inner
    return d


def _normalize_keys(d):
    key_map = {
        "actionType": "action_type",
        "type": "action_type",
        "ActionType": "action_type",
        "taskId": "task_id",
        "developerId": "developer_id",
        "primaryDeveloperId": "primary_developer_id",
        "secondaryDeveloperId": "secondary_developer_id",
        "newPriority": "new_priority",
        "splitRatio": "split_ratio",
    }
    out = {}
    for k, v in d.items():
        nk = key_map.get(k, k)
        out[nk] = v
    return out


def _normalize_action_type(v):
    if v is None:
        return None
    s = str(v).strip().lower().replace("-", "_").replace(" ", "_")
    aliases = {
        "assign": "assign_task",
        "task_assign": "assign_task",
        "rest": "rest_developer",
        "noop": "noop",
        "no_op": "noop",
        "none": "noop",
        "idle": "noop",
        "pair": "pair_program",
        "pairing": "pair_program",
    }
    return aliases.get(s, s)


def _normalize_priority(v):
    if v is None:
        return None
    s = str(v).strip().upper().replace(" ", "_")
    if s in ("LOW", "MEDIUM", "HIGH", "CRITICAL"):
        return s
    return None


def parse_action(text):
    """
    Extract one environment action dict from model text.
    The old regex \\{[^{}]+\\} broke on nested JSON and often grabbed the wrong span.
    """
    raw = _smart_quotes_to_ascii(_strip_json_fences(text or ""))

    candidates = []
    if raw.startswith("{"):
        candidates.append(raw)
    candidates.extend(_balanced_json_slices(raw))

    def try_load(s):
        s = s.strip()
        if not s:
            return None
        s = re.sub(r",(\s*[}\]])", r"\1", s)
        try:
            return json.loads(s)
        except json.JSONDecodeError:
            return None

    for blob in candidates:
        obj = try_load(blob)
        if obj is None:
            continue
        if isinstance(obj, list) and obj:
            obj = obj[0]
        if not isinstance(obj, dict):
            continue
        obj = _unwrap_nested_action(obj)
        obj = _normalize_keys(obj)
        at = _normalize_action_type(obj.get("action_type"))
        if not at:
            continue
        obj["action_type"] = at
        if "new_priority" in obj and obj["new_priority"] is not None:
            npv = _normalize_priority(obj["new_priority"])
            if npv:
                obj["new_priority"] = npv
        for fld in (
            "task_id",
            "developer_id",
            "primary_developer_id",
            "secondary_developer_id",
        ):
            if fld in obj and obj[fld] is not None:
                obj[fld] = str(obj[fld]).strip()
        return obj

    return {"action_type": "noop"}


def _format_shape_reward(completion: str) -> float:
    """Small bonus for parseable JSON + valid action shape (same idea as before)."""
    action = parse_action(completion)
    r = 0.0
    valid_types = {
        "assign_task", "unassign_task", "reprioritize", "rest_developer",
        "split_task", "pair_program", "noop",
    }
    if "action_type" in action:
        r += 0.10
    if action.get("action_type") in valid_types:
        r += 0.15
    if action.get("action_type") == "assign_task":
        if action.get("task_id") and action.get("developer_id"):
            r += 0.30
    if action.get("action_type") == "pair_program":
        if action.get("primary_developer_id"):
            r += 0.20
    if action.get("action_type") == "noop":
        r -= 0.05
    return float(r)


def compute_reward_from_env(session_id: str, final_obs: dict) -> float:
    """Grade + delivery signal from the live HF Space (matches train_grpo.py)."""
    if not final_obs.get("done", False):
        m = final_obs.get("metrics") or {}
        if not isinstance(m, dict):
            m = {}
        total = max(1.0, float(m.get("total_story_points", 1) or 1))
        done_sp = float(m.get("delivered_story_points", 0) or 0)
        return (done_sp / total) * 0.5
    try:
        data = env_client.grade(session_id)["data"]
        weighted_total = float(data.get("weighted_total", 0.0) or 0.0)
        grade_letter = data.get("grade", "F")
        grade_map = {"A+": 1.0, "A": 0.9, "B+": 0.8, "B": 0.7, "C": 0.6, "D": 0.5, "F": 0.0}
        grade_bonus = grade_map.get(grade_letter, 0.0)
        return 0.7 * weighted_total + 0.3 * grade_bonus
    except Exception:
        return 0.0


def _live_step_reward(scenario: str, episode_seed: int, replay_actions_json: str, completion: str) -> float:
    """
    Replay recorded prefix in a fresh session, apply candidate action, return env reward.
    Dataset rows must include scenario, episode_seed, replay_actions_json (see build_dataset).
    """
    fmt = _format_shape_reward(completion)
    try:
        history = json.loads(replay_actions_json) if replay_actions_json else []
    except json.JSONDecodeError:
        history = []

    try:
        reset_resp = env_client.reset(scenario=scenario, seed=int(episode_seed))
        session_id = reset_resp["data"]["session_id"]
        obs = reset_resp["data"]["observation"]
    except Exception:
        return 0.2 * fmt

    for h in history:
        step_action = dict(h)
        step_action["session_id"] = session_id
        try:
            step_resp = env_client.step(session_id, step_action)
            obs = step_resp["data"]["observation"]
        except Exception:
            return 0.15 * fmt

    cand = parse_action(completion)
    cand = {k: v for k, v in cand.items() if v is not None}
    cand["session_id"] = session_id
    try:
        step_resp = env_client.step(session_id, cand)
        obs = step_resp["data"]["observation"]
    except Exception:
        return 0.12 * fmt

    step_r = float(obs.get("reward", 0.0) or 0.0)
    if obs.get("done"):
        term = compute_reward_from_env(session_id, obs)
        return float(max(-0.5, min(2.0, 0.25 * step_r + 0.75 * term + 0.05 * fmt)))
    return float(max(-0.5, min(1.5, step_r + 0.08 * fmt)))


# Share of live env vs format-only (format helps when env is slow / HTTP fails)
GRPO_ENV_REWARD_ALPHA = float(os.environ.get("GRPO_ENV_REWARD_ALPHA", "0.82"))


def grpo_reward_fn(
    prompts,
    completions,
    completion_ids=None,
    trainer_state=None,
    log_extra=None,
    log_metric=None,
    environments=None,
    scenario=None,
    episode_seed=None,
    replay_actions_json=None,
    **kwargs,
):
    """
    Primary signal: one live /step (+ /grader when done) after replaying dataset prefix.
    Secondary: format / schema shaping (keeps learning stable if env is briefly down).
    Requires dataset columns: scenario, episode_seed, replay_actions_json.
    """
    n = len(completions)
    rewards = [0.0] * n

    use_live = (
        scenario is not None
        and episode_seed is not None
        and replay_actions_json is not None
        and len(scenario) == n
        and len(episode_seed) == n
        and len(replay_actions_json) == n
    )

    if not use_live:
        for i, c in enumerate(completions):
            rewards[i] = _format_shape_reward(c)
        return rewards

    for i, c in enumerate(completions):
        live = _live_step_reward(scenario[i], episode_seed[i], replay_actions_json[i], c)
        fmt = _format_shape_reward(c)
        rewards[i] = (1.0 - GRPO_ENV_REWARD_ALPHA) * fmt + GRPO_ENV_REWARD_ALPHA * live
    return rewards


print("✅ Prompt builder, parse_action, env-backed GRPO reward, and format reward ready")

✅ Prompt builder, parse_action, env-backed GRPO reward, and format reward ready


## Cell 7 — Build Training Dataset from Live Environment

In [7]:
from datasets import Dataset

def run_episode_for_dataset(scenario, seed):
    """Roll out one episode; each row includes replay metadata for env-backed GRPO rewards."""
    reset_resp = env_client.reset(scenario=scenario, seed=seed)
    session_id = reset_resp["data"]["session_id"]
    obs = reset_resp["data"]["observation"]
    pairs, done, steps = [], False, 0
    history_actions = []

    while not done and steps < MAX_STEPS_EP:
        prompt = obs_to_prompt(obs, scenario)
        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=512,
        ).to(model.device)
        with torch.no_grad():
            # Keep headroom consistent with GRPO max_completion_length (Colab + Kaggle)
            gen_extra = int(os.environ.get("GEN_EXTRA_TOKENS", "96"))
            gen_max_length = inputs["input_ids"].shape[1] + gen_extra
            out = model.generate(
                **inputs,
                max_length=gen_max_length,
                temperature=0.8,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id,
            )
        completion = tokenizer.decode(
            out[0][inputs["input_ids"].shape[1]:],
            skip_special_tokens=True,
        )
        parsed = parse_action(completion)
        replay_snapshot = json.dumps(history_actions)

        pairs.append({
            "prompt": prompt,
            "completion": completion,
            "scenario": scenario,
            "episode_seed": int(seed),
            "replay_actions_json": replay_snapshot,
        })

        action = dict(parsed)
        action["session_id"] = session_id
        try:
            step_resp = env_client.step(session_id, action)
            obs = step_resp["data"]["observation"]
        except Exception:
            obs = {"done": True, "metrics": {}}

        hist = {k: v for k, v in parsed.items() if k != "session_id"}
        history_actions.append(hist)

        done = obs.get("done", False)
        steps += 1

    return pairs, steps


def build_dataset(n_episodes=MAX_EPISODES):
    all_prompts, all_completions = [], []
    all_scenario, all_episode_seed, all_replay_json = [], [], []
    rng = random.Random(42)
    total_episode_steps = 0
    print(f"  Collecting {n_episodes} episodes (with replay metadata for env rewards)...")
    for i in range(n_episodes):
        scenario = rng.choice(SCENARIOS)
        seed = rng.randint(0, 999)
        try:
            pairs, episode_steps = run_episode_for_dataset(scenario, seed)
            total_episode_steps += episode_steps
            for p in pairs:
                all_prompts.append(p["prompt"])
                all_completions.append(p["completion"])
                all_scenario.append(p["scenario"])
                all_episode_seed.append(p["episode_seed"])
                all_replay_json.append(p["replay_actions_json"])
            avg_steps = total_episode_steps / (i + 1)
            print(
                f"    Episode {i+1:>3}/{n_episodes} | scenario={scenario:<6} | "
                f"seed={seed:>4} | episode_steps={episode_steps:>2} | "
                f"avg_steps={avg_steps:>5.1f} | rows={len(all_prompts)}"
            )
        except Exception as e:
            print(f"    Episode {i+1:>3}/{n_episodes} failed: {e}")

    dataset = Dataset.from_dict({
        "prompt": all_prompts,
        "completion": all_completions,
        "scenario": all_scenario,
        "episode_seed": all_episode_seed,
        "replay_actions_json": all_replay_json,
    })
    print(f"\n✅ Dataset: {len(dataset)} steps from {n_episodes} episodes (columns for live GRPO reward)")
    return dataset


dataset = build_dataset(n_episodes=MAX_EPISODES)

/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API i

    Episode   1/50 | scenario=hard   | seed= 114 | episode_steps=33 | avg_steps= 33.0 | rows=33
    Episode   2/50 | scenario=easy   | seed= 759 | episode_steps= 3 | avg_steps= 18.0 | rows=36
    Episode   3/50 | scenario=medium | seed= 250 | episode_steps= 3 | avg_steps= 13.0 | rows=39
    Episode   4/50 | scenario=easy   | seed= 142 | episode_steps= 1 | avg_steps= 10.0 | rows=40
    Episode   5/50 | scenario=hard   | seed= 104 | episode_steps=29 | avg_steps= 13.8 | rows=69
    Episode   6/50 | scenario=hard   | seed= 758 | episode_steps=29 | avg_steps= 16.3 | rows=98
    Episode   7/50 | scenario=hard   | seed=  89 | episode_steps=27 | avg_steps= 17.9 | rows=125
    Episode   8/50 | scenario=hard   | seed= 432 | episode_steps=26 | avg_steps= 18.9 | rows=151
    Episode   9/50 | scenario=easy   | seed=  30 | episode_steps= 2 | avg_steps= 17.0 | rows=153
    Episode  10/50 | scenario=easy   | seed= 223 | episode_steps= 4 | avg_steps= 15.7 | rows=157
    Episode  11/50 | scenario=easy  

## Cell 8 — GRPO Training

In [ ]:
import os
import warnings
from trl import GRPOConfig, GRPOTrainer
from transformers import logging as hf_logging
from transformers.trainer_callback import PrinterCallback, TrainerCallback

# Silence repetitive transformers logs/warnings in notebook output.
hf_logging.set_verbosity_error()

# Hide annoying transformers FutureWarnings (like AttentionMaskConverter)
warnings.simplefilter("ignore", category=FutureWarning)
import os; os.environ["PYTHONWARNINGS"] = "ignore"


def _to_float(x) -> float:
    try:
        return float(x)
    except Exception:
        return 0.0


def _std_for_mean_key(logs: dict, mean_key: str) -> float:
    candidates: tuple[str, ...]
    if mean_key == "rewards/grpo_reward_fn/mean":
        candidates = ("rewards/grpo_reward_fn/std", "rewards/std")
    elif mean_key == "rewards/mean":
        candidates = ("rewards/std",)
    else:
        candidates = ("rewards/std",)
    for k in candidates:
        if k in logs and logs[k] is not None:
            return _to_float(logs[k])
    return 0.0


def _pick_mean_reward(logs: dict) -> tuple[float, str]:
    if REWARD_LOG_KEY and REWARD_LOG_KEY in logs and logs[REWARD_LOG_KEY] is not None:
        return _to_float(logs[REWARD_LOG_KEY]), REWARD_LOG_KEY
    # Sensible fallbacks in priority order
    for k in (
        "rewards/grpo_reward_fn/mean",
        "rewards/mean",
        "rewards/reward",
        "reward",
    ):
        if k in logs and logs[k] is not None:
            return _to_float(logs[k]), k
    return 0.0, "missing"


# After-train plots + a clean "HF-wide" 3-column live table.
# Default: track the GRPO reward function mean (the per-reward-func row in full HF tables).
# Override via env: REWARD_LOG_KEY=rewards/mean
REWARD_LOG_KEY = os.environ.get("REWARD_LOG_KEY", "rewards/grpo_reward_fn/mean").strip()
# 1 = print the wide table on every log; 0 = quiet training + print a clean table at the end
LIVE_TRAINING_TABLE = int(os.environ.get("LIVE_TRAINING_TABLE", "0"))

training_log = {"step": [], "loss": [], "reward_mean": [], "reward_std": []}


def print_step_loss_table_from_log(also_reward: bool = False) -> None:
    """Kaggle/Colab friendly: Step vs Training Loss (optionally + GRPO reward)."""
    if not training_log.get("step"):
        print("No training_log collected yet (training didn't log anything).")
        return

    sep = "  "
    if also_reward:
        h = f"{'Step':>5s}{sep}{'Training Loss':>15s}{sep}{'Reward(GRPO)':>18s}"
    else:
        h = f"{'Step':>5s}{sep}{'Training Loss':>15s}"
    print("\n" + h + "\n" + ("-" * len(h)))

    for i, step in enumerate(training_log["step"]):
        loss = _to_float(training_log["loss"][i])
        if abs(loss) < 1e-3 and loss != 0.0:
            loss_s = f"{loss:.3e}"
        else:
            loss_s = f"{loss: .6f}"

        if also_reward:
            m = _to_float(training_log["reward_mean"][i])
            s = _to_float(training_log["reward_std"][i])
            r_s = f"{m: .6f}±{s:.4f}" if s > 0 else f"{m: .6f}"
            print(f"{int(step):>5d}{sep}{loss_s:>15s}{sep}{r_s:>18s}")
        else:
            print(f"{int(step):>5d}{sep}{loss_s:>15s}")


class RewardTrackingCallback(TrainerCallback):
    """Records Trainer logs. Live printing is optional (see LIVE_TRAINING_TABLE)."""

    def __init__(self):
        super().__init__()
        self._header_printed = False
        self._last_mean_key = None

    def _print_header_live(self):
        if self._header_printed:
            return
        sep = " " * 2
        line1 = f"{'Step':>5s}{sep}{'Training Loss':>15s}{sep}{'Reward(GRPO)':>20s}"
        line2 = "-" * len(line1)
        print("\n" + line1 + "\n" + line2)
        self._header_printed = True

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs or "loss" not in logs:
            return

        step = int(state.global_step)
        loss = _to_float(logs.get("loss", 0.0))

        mean, used_key = _pick_mean_reward(logs)
        self._last_mean_key = used_key
        r_std = _std_for_mean_key(logs, used_key) if used_key != "missing" else 0.0

        training_log["step"].append(step)
        training_log["loss"].append(loss)
        training_log["reward_mean"].append(mean)
        training_log["reward_std"].append(r_std)

        if not LIVE_TRAINING_TABLE:
            return

        self._print_header_live()
        sep = " " * 2
        if abs(loss) < 1e-3 and loss != 0.0:
            loss_s = f"{loss:.3e}"
        else:
            loss_s = f"{loss:.6f}"

        reward_s = f"{mean: .6f}±{r_std:.4f}" if r_std > 0 else f"{mean: .6f}"
        print(f"{step:>5d}{sep}{loss_s:>15s}{sep}{reward_s:>20s}")

# One-click mode switch:
# - "fast": Kaggle iteration/debug (minutes)
# - "full": higher-quality final run (can take hours)
TRAIN_MODE = "fast"  # change to "full" when you want final training
MAX_TRAIN_SAMPLES = int(os.environ.get("MAX_TRAIN_SAMPLES", "600"))

CFG_FAST = dict(
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    # Slightly higher G reduces reward variance (works the same on Colab + Kaggle; uses more VRAM/time)
    num_generations=4,
    max_prompt_length=512,
    # If this is too small, TRL will clip every completion (clipped_ratio=1.0) which inflates reward noise.
    max_completion_length=192,
    learning_rate=5e-6,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    logging_steps=5,
    save_steps=200,
    output_dir=OUTPUT_DIR,
    report_to="wandb",
    # A bit more KL anchoring helps stability on both runtimes when rewards are noisy.
    beta=0.06,
    use_vllm=False,
    remove_unused_columns=False,
    disable_tqdm=False,
)

CFG_FULL = dict(
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_generations=8,
    max_prompt_length=512,
    max_completion_length=192,
    learning_rate=5e-6,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    logging_steps=5,
    save_steps=50,
    output_dir=OUTPUT_DIR,
    report_to="wandb",
    beta=0.06,
    use_vllm=False,
    remove_unused_columns=False,
    disable_tqdm=False,
)

if TRAIN_MODE not in {"fast", "full"}:
    raise ValueError("TRAIN_MODE must be 'fast' or 'full'")

cfg = CFG_FAST if TRAIN_MODE == "fast" else CFG_FULL
cfg["report_to"] = "wandb"
cfg["run_name"] = "openenv-pm-grpo"
config = GRPOConfig(**cfg)

dataset_train = dataset
if TRAIN_MODE == "fast" and len(dataset) > MAX_TRAIN_SAMPLES:
    dataset_train = dataset.shuffle(seed=42).select(range(MAX_TRAIN_SAMPLES))

print(
    f"TRAIN_MODE={TRAIN_MODE} | rows={len(dataset_train)}/{len(dataset)} | "
    f"num_generations={cfg['num_generations']}"
)
print(
    "Live env-backed reward is expensive (HTTP reset/replay/step during training). "
    "Use TRAIN_MODE='fast' for Kaggle testing and TRAIN_MODE='full' for final run."
)

# --- Runtime explanation helpers (so tqdm max_steps is understandable) ---
try:
    import torch

    n_gpu = torch.cuda.device_count() if torch.cuda.is_available() else 0
except Exception:
    n_gpu = 0

N = int(len(dataset_train))
B = int(config.per_device_train_batch_size)
G = int(config.gradient_accumulation_steps)
E = int(config.num_train_epochs)
world = max(1, n_gpu)  # HF multi-GPU DataParallel/DDP style global batch proxy

# Effective "optimizer step" size per microbatch cycle is roughly: world * B * G
# Steps per epoch ~ ceil(N / (world * B * G))  (HF rounds; exact depends on drop_last etc.)
import math

micro_bs = world * B * G
steps_per_epoch = math.ceil(N / max(1, micro_bs)) if N > 0 else 0
expected_max_steps = steps_per_epoch * E

print("\n=== TRAINING STEP MATH (approx) ===")
print(f"dataset_train rows (N)      : {N}")
print(f"num_train_epochs (E)        : {E}")
print(f"per_device_train_batch_size : {B}")
print(f"gradient_accumulation_steps : {G}")
print(f"visible GPUs (torch)        : {n_gpu}  (world~{world})")
print(f"~micro batch size (w*B*G)   : {micro_bs}")
print(f"~steps_per_epoch (ceil(N/…)): {steps_per_epoch}")
print(f"~expected max_steps (×E)     : {expected_max_steps}")
print("Note: GRPO/TRL can adjust scheduling internally; the tqdm total is the Trainer’s computed max_steps.\n")

trainer = GRPOTrainer(
    model=model,
    tokenizer=tokenizer,
    reward_funcs=[grpo_reward_fn],
    args=config,
    train_dataset=dataset_train,
    callbacks=[RewardTrackingCallback()],
)

# HF default prints a very wide metric table; we replace it with a 3-column wide table.
trainer.remove_callback(PrinterCallback)

print("Starting GRPO training...")
trainer.train()

# Kaggle-style summary table (like your reference screenshot)
print_step_loss_table_from_log(also_reward=False)
try:
    n_points = len(training_log.get("step", []))
    g = int(getattr(config, "num_generations", 1) or 1)
    print(f"\nTraining done. Reward samples: {n_points * g}")
except Exception:
    print("\nTraining done.")

model.save_pretrained(f"{OUTPUT_DIR}/lora_adapter")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/lora_adapter")
print(f"\n✅ LoRA adapter saved to {OUTPUT_DIR}/lora_adapter")

TRAIN_MODE=fast | rows=522/522 | num_generations=4
Live env-backed reward is expensive (HTTP reset/replay/step during training). Use TRAIN_MODE='fast' for Kaggle testing and TRAIN_MODE='full' for final run.

=== TRAINING STEP MATH (approx) ===
dataset_train rows (N)      : 522
num_train_epochs (E)        : 1
per_device_train_batch_size : 2
gradient_accumulation_steps : 2
visible GPUs (torch)        : 2  (world~2)
~micro batch size (w*B*G)   : 8
~steps_per_epoch (ceil(N/…)): 66
~expected max_steps (×E)     : 66
Note: GRPO/TRL can adjust scheduling internally; the tqdm total is the Trainer’s computed max_steps.

Starting GRPO training...


wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

  1


wandb: You chose 'Create a W&B account'
wandb: Create an account here: https://wandb.ai/authorize?signup=true&ref=models
wandb: After creating your account, create a new API key and store it securely.
wandb: Paste your API key and hit enter:

  ········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: dharshansriram-r (dharshansriram-r-ciet) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference, mcp, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / grpo_reward_fn / mean,rewards / grpo_reward_fn / std
5,0.000000,-0.066383,0.018767,192.000000,192.000000,192.000000,1.000000,0.000000,0.000000,0.000000,0.000009,-0.066383,0.018767
10,0.000001,0.020160,0.018018,192.000000,192.000000,192.000000,1.000000,0.000000,0.000000,0.000000,0.000011,0.020160,0.018018
15,0.000001,0.025986,0.051335,192.000000,192.000000,192.000000,1.000000,0.000000,0.000000,0.000000,0.000019,0.025986,0.051335
20,0.000001,0.034694,0.069048,192.000000,192.000000,192.000000,1.000000,0.000000,0.000000,0.000000,0.000024,0.034694,0.069048
25,0.000001,-0.069726,0.040395,192.000000,192.000000,192.000000,1.000000,0.000000,0.000000,0.000000,0.000025,-0.069726,0.040395
30,0.000013,0.016294,0.062382,192.000000,192.000000,192.000000,1.000000,0.000000,0.000000,0.000000,0.000216,0.016294,0.062382
35,0.000006,0.020196,0.019847,192.000000,192.000000,192.000000,1.000000,0.000000,0.000000,0.000000,0.000101,0.020196,0.019847
40,0.000010,-0.132296,0.050821,192.000000,192.000000,192.000000,1.000000,0.000000,0.000000,0.000000,0.000170,-0.132296,0.050821
45,0.000007,0.042665,0.012024,192.000000,192.000000,192.000000,1.000000,0.000000,0.000000,0.000000,0.000115,0.042665,0.012024
50,0.000023,0.004374,0.021770,192.000000,192.000000,192.000000,1.000000,0.000000,0.000000,0.000000,0.000387,0.004374,0.021770


## Cell 9 — 📈 Training Curves (Loss + Reward)

In [ ]:
def plot_training_curves(log, save_path=f"{OUTPUT_DIR}/training_curves.png"):
    steps      = log["step"]
    losses     = log["loss"]
    rewards    = log["reward_mean"]
    reward_std = log["reward_std"]

    if not steps:
        print("No training logs yet — run training first")
        return

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("PM-GRPO Training — Adaptive AI Project Manager",
                 fontsize=14, fontweight="bold")

    ax1.plot(steps, losses, color="#2563eb", linewidth=2, label="Training Loss")
    ax1.fill_between(steps, losses, alpha=0.1, color="#2563eb")
    ax1.set_xlabel("Training Step")
    ax1.set_ylabel("Loss")
    ax1.set_title("Training Loss")
    ax1.grid(alpha=0.3)
    ax1.legend()

    r_arr   = np.array(rewards)
    std_arr = np.array(reward_std)
    ax2.plot(steps, r_arr, color="#16a34a", linewidth=2, label="Mean Reward")
    ax2.fill_between(steps, r_arr - std_arr, r_arr + std_arr,
                     alpha=0.15, color="#16a34a", label="±1 std")
    ax2.set_xlabel("Training Step")
    ax2.set_ylabel("Reward")
    ax2.set_title("Episode Reward (GRPO)")
    ax2.grid(alpha=0.3)
    ax2.legend()

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"✅ Saved → {save_path}")


plot_training_curves(training_log)

## Cell 10 — Evaluate Trained Agent (AFTER Training)

In [ ]:
# ✅ FIX A: Switch Unsloth model to fast inference mode before evaluation
# Without this, trained Unsloth model stays in training mode → slow + garbled output
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)
print("✅ Model in inference mode.")


def choose_fallback_action(obs):
    tasks = obs.get("tasks", [])
    devs  = obs.get("developers", [])

    candidate_tasks = [
        t for t in tasks
        if t.get("status") in ("ready", "in_progress") and t.get("remaining_points", 1) > 0
    ]
    candidate_tasks.sort(key=lambda t: (
        0 if t.get("priority") == "CRITICAL" else 1,
        -t.get("business_value", 0),
        t.get("deadline_step", 9999),
    ))

    available_devs = [d for d in devs if d.get("available", False)]
    available_devs.sort(key=lambda d: d.get("fatigue", 1.0))

    if candidate_tasks and available_devs:
        return {
            "action_type": "assign_task",
            "task_id":     candidate_tasks[0].get("id"),
            "developer_id": available_devs[0].get("id"),
        }

    busy_devs = [d for d in devs if d.get("current_tasks")]
    busy_devs.sort(key=lambda d: d.get("fatigue", 0.0), reverse=True)
    if busy_devs:
        return {"action_type": "rest_developer", "developer_id": busy_devs[0].get("id")}

    return {"action_type": "noop"}


def sanitize_action(action, obs):
    valid_types = {
        "assign_task", "unassign_task", "reprioritize",
        "rest_developer", "split_task", "pair_program", "noop",
    }
    if not isinstance(action, dict) or action.get("action_type") not in valid_types:
        return choose_fallback_action(obs), "invalid_action_type"
    at = action.get("action_type")
    if at == "assign_task" and (not action.get("task_id") or not action.get("developer_id")):
        return choose_fallback_action(obs), "bad_assign_payload"
    if at == "rest_developer" and not action.get("developer_id"):
        return choose_fallback_action(obs), "bad_rest_payload"
    return action, None


def run_episode_trained(scenario, seed, verbose=False):
    reset_resp = env_client.reset(scenario=scenario, seed=seed)
    session_id = reset_resp["data"]["session_id"]
    obs        = reset_resp["data"]["observation"]

    done, steps       = False, 0
    fallbacks_used    = 0
    step_errors       = 0
    consecutive_errors = 0          # ✅ FIX C: track consecutive errors, not global

    while not done and steps < MAX_STEPS_EP:
        prompt = obs_to_prompt(obs, scenario)
        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=512,
        ).to(model.device)

        # ✅ FIX B: proper inference generation
        with torch.no_grad():
            gen_extra = int(os.environ.get("GEN_EXTRA_TOKENS", "128"))
            out = model.generate(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"],   # explicit mask
                max_new_tokens=gen_extra,
                temperature=0.3,
                do_sample=True,                            # was False — greedy ignored temperature
                repetition_penalty=1.1,                    # prevent JSON key repetition
                pad_token_id=tokenizer.eos_token_id,
            )

        completion = tokenizer.decode(
            out[0][inputs["input_ids"].shape[1]:],
            skip_special_tokens=True,
        )

        raw_action          = parse_action(completion)
        action, reason      = sanitize_action(raw_action, obs)
        if reason is not None:
            fallbacks_used += 1

        try:
            step_resp          = env_client.step(session_id, action)
            obs                = step_resp["data"]["observation"]
            consecutive_errors = 0                         # reset on success
        except Exception as e:
            step_errors       += 1
            consecutive_errors += 1
            if verbose:
                print(f"    step error at step {steps}: {e}")

            # ✅ FIX C: use heuristic fallback + retry; only hard-kill after 3 consecutive errors
            if consecutive_errors >= 3:
                if verbose:
                    print("    3 consecutive step errors — ending episode early")
                break                                      # was: obs = {"done": True} kills grade
            # try the safe fallback action before giving up on this step
            try:
                fallback_act = choose_fallback_action(obs)
                fallback_act["session_id"] = session_id
                step_resp    = env_client.step(session_id, fallback_act)
                obs          = step_resp["data"]["observation"]
                consecutive_errors = 0
                fallbacks_used += 1
            except Exception:
                obs = obs  # keep last known obs and carry on

        done  = obs.get("done", False)
        steps += 1

    try:
        g = env_client.grade(session_id)["data"]
        return g.get("weighted_total", 0.0), g.get("grade", "F"), {
            "steps":       steps,
            "fallbacks":   fallbacks_used,
            "step_errors": step_errors,
        }
    except Exception as e:
        if verbose:
            print(f"    grade error: {e}")
        return 0.0, "F", {
            "steps":       steps,
            "fallbacks":   fallbacks_used,
            "step_errors": step_errors + 1,
        }


print("Evaluating TRAINED agent...")
print("-" * 70)
trained_scores = {}
for scenario in SCENARIOS:
    scores, grades = [], []
    total_steps, total_fallbacks, total_errors = 0, 0, 0

    for seed in range(EVAL_SEEDS):
        score, grade, stats = run_episode_trained(scenario, seed, verbose=False)
        scores.append(score)
        grades.append(grade)
        total_steps     += stats["steps"]
        total_fallbacks += stats["fallbacks"]
        total_errors    += stats["step_errors"]

    trained_scores[scenario] = scores
    mean          = sum(scores) / len(scores)
    fallback_rate = (total_fallbacks / max(1, total_steps)) * 100
    print(
        f"  {scenario:<8} | mean={mean:.3f} | grades={grades} | "
        f"fallback_rate={fallback_rate:.1f}% | step_errors={total_errors}"
    )

print("\nEvaluation complete.")


## Cell 11 — 📊 Before vs After Comparison

In [ ]:
def plot_before_after(baseline, trained, save_path=f"{OUTPUT_DIR}/before_after.png"):
    scenarios   = list(baseline.keys())
    base_means  = [sum(baseline[s]) / len(baseline[s]) for s in scenarios]
    train_means = [sum(trained[s])  / len(trained[s])  for s in scenarios]
    base_stds   = [np.std(baseline[s]) for s in scenarios]
    train_stds  = [np.std(trained[s])  for s in scenarios]

    x     = np.arange(len(scenarios))
    width = 0.35
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("Adaptive AI Project Manager — Before vs After GRPO Training",
                 fontsize=13, fontweight="bold")

    ax1.bar(x - width/2, base_means,  width, label="Baseline (Heuristic)",
            color="#94a3b8", yerr=base_stds,  capsize=4)
    ax1.bar(x + width/2, train_means, width, label="Trained (GRPO)",
            color="#2563eb", yerr=train_stds, capsize=4)
    for i, (bm, tm) in enumerate(zip(base_means, train_means)):
        delta = tm - bm
        color = "#16a34a" if delta >= 0 else "#dc2626"
        sign  = "+" if delta >= 0 else ""
        ax1.text(x[i] + width/2, tm + 0.02,
                 f"{sign}{delta:.3f}", ha="center", fontsize=10,
                 fontweight="bold", color=color)
    ax1.set_xticks(x)
    ax1.set_xticklabels([s.capitalize() for s in scenarios])
    ax1.set_ylabel("Weighted Score (0–1)")
    ax1.set_title("Score by Scenario")
    ax1.set_ylim(0, 1.15)
    ax1.legend()
    ax1.grid(axis="y", alpha=0.3)

    colors = ["#f59e0b", "#10b981", "#8b5cf6"]
    for sc, col in zip(scenarios, colors):
        n = len(baseline[sc])
        xs = list(range(n))
        ax2.plot(xs, baseline[sc], "o--", color=col, alpha=0.5,
                 label=f"{sc} baseline", markersize=6)
        ax2.plot(xs, trained[sc],  "s-",  color=col,
                 label=f"{sc} trained",  markersize=8, linewidth=2)
    ax2.set_xlabel("Seed")
    ax2.set_ylabel("Weighted Score (0–1)")
    ax2.set_title("Per-Seed Comparison")
    ax2.legend(fontsize=9, ncol=2)
    ax2.grid(alpha=0.3)
    ax2.set_ylim(0, 1.15)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"✅ Saved → {save_path}")


plot_before_after(baseline_scores, trained_scores)

print("\n" + "=" * 58)
print(f"  {'Scenario':<10} | {'Baseline':>9} | {'Trained':>9} | {'Delta':>8}")
print("=" * 58)
for sc in SCENARIOS:
    bm = sum(baseline_scores[sc]) / len(baseline_scores[sc])
    tm = sum(trained_scores[sc])  / len(trained_scores[sc])
    d  = tm - bm
    sign = "+" if d >= 0 else ""
    print(f"  {sc:<10} | {bm:>9.4f} | {tm:>9.4f} | {sign}{d:>7.4f}")
print("=" * 58)

## Cell 12 — 📉 Episode Reward Improvement Over Time

In [ ]:
def plot_episode_rewards(episode_rewards, save_path=f"{OUTPUT_DIR}/episode_rewards.png"):
    rewards  = np.array(episode_rewards)
    n        = len(rewards)
    window   = max(1, n // 10)
    smoothed = np.convolve(rewards, np.ones(window)/window, mode="valid")
    xs_raw   = np.arange(n)
    xs_sm    = np.arange(len(smoothed)) + window // 2

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.scatter(xs_raw, rewards, alpha=0.25, s=12, color="#93c5fd", label="Episode reward")
    ax.plot(xs_sm, smoothed, color="#1d4ed8", linewidth=2.5,
            label=f"Smoothed (window={window})")

    first_half  = rewards[:n//2].mean()
    second_half = rewards[n//2:].mean()
    ax.axhline(first_half,  color="#f59e0b", linestyle="--", alpha=0.8,
               label=f"First-half mean:  {first_half:.3f}")
    ax.axhline(second_half, color="#16a34a", linestyle="--", alpha=0.8,
               label=f"Second-half mean: {second_half:.3f}")

    delta = second_half - first_half
    sign  = "+" if delta >= 0 else ""
    ax.text(n * 0.75, max(rewards) * 0.95,
            f"Improvement: {sign}{delta:.3f}",
            fontsize=13, fontweight="bold",
            color="#16a34a" if delta >= 0 else "#dc2626")

    ax.set_xlabel("Episode")
    ax.set_ylabel("Episode Reward (Weighted Score)")
    ax.set_title("Episode Reward Over Training", fontsize=12, fontweight="bold")
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"✅ Saved → {save_path}")


print("Collecting episode reward curve (trained agent, 30 episodes)...")
ep_rewards = []
rng2 = random.Random(99)
for i in range(30):
    sc   = rng2.choice(SCENARIOS)
    seed = rng2.randint(0, 200)
    # ✅ FIX 3: run_episode_trained returns 3 values (score, grade, stats)
    score, _grade, _stats = run_episode_trained(sc, seed)
    ep_rewards.append(score)
    if (i + 1) % 5 == 0:
        print(f"  {i+1}/30 episodes collected")

plot_episode_rewards(ep_rewards)

## Cell 13 — 📉 Per-Rubric Reward Breakdown

In [ ]:
def collect_rubric_scores(scenario="medium", n_seeds=3):
    rubric_totals = defaultdict(list)
    for seed in range(n_seeds):
        reset_resp = env_client.reset(scenario=scenario, seed=seed)
        session_id = reset_resp["data"]["session_id"]
        obs        = reset_resp["data"]["observation"]
        done, steps = False, 0
        while not done and steps < MAX_STEPS_EP:
            prompt = obs_to_prompt(obs, scenario)
            inputs = tokenizer(prompt, return_tensors="pt",
                               truncation=True, max_length=512).to(model.device)
            with torch.no_grad():
                gen_extra = int(os.environ.get("GEN_EXTRA_TOKENS", "96"))
                gen_max_length = inputs["input_ids"].shape[1] + gen_extra
                out = model.generate(**inputs, max_length=gen_max_length, temperature=0.3,
                                     do_sample=True, pad_token_id=tokenizer.eos_token_id)
            completion = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:],
                                          skip_special_tokens=True)
            action = parse_action(completion)
            action["session_id"] = session_id
            try:
                step_resp = env_client.step(session_id, action)
                obs = step_resp["data"]["observation"]
                rubric_bd = obs.get("info", {}).get("rubric_breakdown", {})
                for name, data in rubric_bd.get("rubrics", {}).items():
                    rubric_totals[name].append(data.get("weighted", 0))
            except:
                obs = {"done": True}
            done = obs.get("done", False)
            steps += 1
    return {k: sum(v)/len(v) if v else 0 for k, v in rubric_totals.items()}


def plot_rubric_breakdown(scores, save_path=f"{OUTPUT_DIR}/rubric_breakdown.png"):
    names  = list(scores.keys())
    values = [scores[n] for n in names]
    colors = ["#16a34a" if v >= 0 else "#dc2626" for v in values]

    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.barh(names, values, color=colors, edgecolor="white")
    ax.axvline(0, color="#374151", linewidth=1)
    ax.set_xlabel("Mean Weighted Reward per Step")
    ax.set_title("Per-Rubric Reward Breakdown (Trained Agent)", fontweight="bold")
    ax.grid(axis="x", alpha=0.3)
    for bar, val in zip(bars, values):
        sign = "+" if val >= 0 else ""
        ax.text(val + (0.001 if val >= 0 else -0.001),
                bar.get_y() + bar.get_height()/2,
                f"{sign}{val:.4f}", va="center",
                ha="left" if val >= 0 else "right", fontsize=9)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"✅ Saved → {save_path}")


print("Collecting rubric breakdown from trained agent...")
rubric_scores = collect_rubric_scores(scenario="medium", n_seeds=3)
plot_rubric_breakdown(rubric_scores)

## Cell 14 — Push Model to HuggingFace Hub

In [ ]:
# ✅ FIX 2: Adaptive HF_TOKEN — use platform secrets if available
import os, sys

HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN")

if HF_TOKEN is None:
    try:
        if IS_KAGGLE:
            from kaggle_secrets import UserSecretsClient
            HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
            print("HF_TOKEN loaded from Kaggle secrets.")
        elif IS_COLAB:
            from google.colab import userdata
            HF_TOKEN = userdata.get("HF_TOKEN")
            print("HF_TOKEN loaded from Colab secrets.")
    except Exception as e:
        print(f"Could not load HF_TOKEN from secrets: {e}")
        HF_TOKEN = None

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN is not set. "
        "Add it as a secret named 'HF_TOKEN' in Kaggle/Colab, "
        "or set os.environ['HF_TOKEN'] before this cell."
    )

from huggingface_hub import login
login(token=HF_TOKEN)
model.push_to_hub("dharshansriram/openenv-pr-ai", token=HF_TOKEN)
tokenizer.push_to_hub("dharshansriram/openenv-pr-ai", token=HF_TOKEN)
print("Pushed to: https://huggingface.co/dharshansriram/openenv-pr-ai")


## Cell 15 — Summary + Files Generated

In [ ]:
print("=" * 60)
print("  TRAINING COMPLETE — SUMMARY")
print("=" * 60)
for sc in SCENARIOS:
    bm   = sum(baseline_scores[sc]) / len(baseline_scores[sc])
    tm   = sum(trained_scores[sc])  / len(trained_scores[sc])
    d    = tm - bm
    sign = "+" if d >= 0 else ""
    bar  = chr(9608) * int(tm * 20) + chr(9617) * (20 - int(tm * 20))
    print(f"  {sc:<8} baseline={bm:.3f} → trained={tm:.3f}  [{sign}{d:.3f}]  {bar}")

print("\nFiles saved:")
for fname in ["training_curves.png", "before_after.png",
              "rubric_breakdown.png", "episode_rewards.png"]:
    path = f"{OUTPUT_DIR}/{fname}"
    size = os.path.getsize(path) // 1024 if os.path.exists(path) else 0
    status = f"✅ {size}KB" if os.path.exists(path) else "❌ missing"
    print(f"  {status}  {path}")

print("\nNext steps:")
print("  1. Download the 4 PNG plots from the files panel")
print("  2. Commit them to your repo")
print("  3. Embed in README.md: ![curve](./training_curves.png)")
print("  4. Link this Colab notebook URL from your README")

## Post-training: hard scenario quick eval + reward smoothing

Run this **after** the training cell. It mirrors the "before/after" + `np.convolve` style plots from reference notebooks.

In [ ]:
# Quick hard-scenario check (5 rollouts) — requires `run_episode_trained` from the eval cell
print("Running post-training eval (hard, n=5)...")

try:
    _ = baseline_scores  # noqa: F821
except NameError:
    baseline_scores = None

after_scores = []
for seed in range(5):
    score, _grade, _stats = run_episode_trained("hard", seed, verbose=False)
    after_scores.append(float(score))
after_mean = sum(after_scores) / max(1, len(after_scores))

if baseline_scores is not None and "hard" in baseline_scores:
    base_list = [float(x) for x in baseline_scores["hard"]]
    base_mean = sum(base_list) / len(base_list)
    print(f"Before (hard, heuristic demo): {base_list} -> {base_mean:.4f}")
else:
    base_mean = None
    print("Before (hard): (run baseline cell first to populate baseline_scores) -> N/A")

print(f"After  (hard, trained policy) : {after_scores} -> {after_mean:.4f}")
if base_mean is not None:
    print(f"Improvement: {after_mean - base_mean:+.4f}")


# Reward smoothing like reference notebooks: np.convolve moving average
import numpy as np
import matplotlib.pyplot as plt

reward_log = [float(x) for x in training_log.get("reward_mean", [])]
window = 10
if len(reward_log) >= window:
    kernel = np.ones(window, dtype=float) / float(window)
    smoothed = np.convolve(reward_log, kernel, mode="valid")
else:
    smoothed = np.array(reward_log, dtype=float)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(reward_log, alpha=0.3, color="steelblue", label="reward (logged mean)")
if len(smoothed):
    axes[0].plot(
        range(window - 1, window - 1 + len(smoothed)),
        smoothed,
        color="darkblue",
        linewidth=2,
        label=f"MA({window})",
    )
axes[0].set_title("GRPO reward (from training log)")
axes[0].set_xlabel("log index")
axes[0].set_ylabel("reward")
axes[0].grid(alpha=0.3)
axes[0].legend()

if training_log.get("loss"):
    axes[1].plot(
        [int(s) for s in training_log["step"]],
        [float(x) for x in training_log["loss"]],
        color="#2563eb",
    )
    axes[1].set_title("Training loss")
    axes[1].set_xlabel("step")
    axes[1].set_ylabel("loss")
    axes[1].grid(alpha=0.3)
else:
    axes[1].text(0.1, 0.5, "No loss log captured", transform=axes[1].transAxes)

plt.tight_layout()
plt.show()